### **Setup**

In [ ]:
from IPython.display import clear_output

#### **Installation Folder**

In [ ]:
!rm -rf ~/.cache/pip
!mkdir -p /content/pip_tmp
!mkdir -p /content/python_packages

%env TMPDIR=/content/pip_tmp
%env PIP_CACHE_DIR=/content/pip_tmp

clear_output()

#### **Remove all pip packages**

In [ ]:
# !pip uninstall -y -r <(pip freeze)
# clear_output()

#### **Install Packages**

In [ ]:
!pip install --no-cache-dir --target=/content/python_packages --upgrade \
    demucs opencv-python ultralytics scenedetect mediapipe faster-whisper lapx pandas python_speech_features easyocr

clear_output()

### **Imports**

In [ ]:
import sys

try:
    sys.path.remove('/content/python_packages')
    sys.path.insert(0, '/content/python_packages')
except:
    sys.path.insert(0, '/content/python_packages')

In [ ]:
import os
import re
import gc
import ast
import cv2
import json
import math
import torch
import shutil
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import easyocr
from ultralytics import YOLO
from faster_whisper import WhisperModel

# Shot Detection
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector

# Face Detection and Embedding
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

# TalkNet
import torch
import torch.nn as nn
from pathlib import Path
from typing import Any, Dict, List
import cv2
import numpy as np
import pandas as pd
import python_speech_features
from scipy import signal
from scipy.interpolate import interp1d
from scipy.io import wavfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import MultiheadAttention

### **Classes & Functions**

In [ ]:
if torch.cuda.is_available():
    ACTIVE_DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    ACTIVE_DEVICE= "mps"
else:
    ACTIVE_DEVICE = "cpu"

ACTIVE_DEVICE

'cuda'

#### **Helper**

In [ ]:
def clean_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    elif hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()

def init_input_video(input_video_path: str) -> Dict:
    input_video_path = Path(input_video_path)
    if not input_video_path.exists():
        raise FileNotFoundError(f"Input video not found: {input_video_path}")

    # [WIP] Standardize -> .mov to .mp4

    # Probe the video
    cap = cv2.VideoCapture(input_video_path)
    if cap.isOpened():
        fps = float(cap.get(cv2.CAP_PROP_FPS) or None)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or None)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or None)
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or None)
        duration = (
            float(total_frames / fps)
            if fps > 0 and total_frames > 0
            else None
        )
    cap.release()

    # Create a working directory
    Path(f"./working/{input_video_path.stem}").mkdir(exist_ok=True, parents=True)

    return {
        "path": str(input_video_path),

        "parent_dir": str(input_video_path.parent),
        "filename": str(input_video_path.stem),
        "extension": str(input_video_path.suffix),

        "fps": fps,
        "total_frames": total_frames,
        "width": width,
        "height": height,
        "duration": duration
    }

#### **Seperate Audio & Video**

In [ ]:
def separate_audio(input_video: Dict) -> str:
    output_path = f"./working/{input_video['filename']}/audio_only.wav"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            input_video["path"],
            "-vn",
            "-acodec",
            "pcm_s16le",
            "-ar",
            "16000", # 44.1 -> 16
            "-ac",
            "2",
            output_path,
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )
    return output_path

def separate_video(input_video: Dict) -> str:
    output_path = f"./working/{input_video['filename']}/video_only.mp4"
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            input_video["path"],
            "-an",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-movflags",
            "+faststart",
            output_path,
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )
    return output_path

#### **Shot Detection**

In [ ]:
def shot_detection(input_video: Dict, threshold: float = 27.0) -> pd.DataFrame:
    if Path(f"./working/{input_video['filename']}/video_only.mp4").exists():
        video = f"./working/{input_video['filename']}/video_only.mp4"
    else:
        video = input_video["path"]

    video = open_video(video)
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=threshold))
    scene_manager.detect_scenes(video)
    scene_list = scene_manager.get_scene_list()

    if not scene_list:
        scene_list = [(video.base_timecode, video.duration)]

    rows = []
    for idx, (start, end) in enumerate(scene_list):
        rows.append({
            "shot_id": idx,

            "start_frame": start.frame_num,
            "end_frame": end.frame_num,
            "duration_frame": end.frame_num - start.frame_num,

            "start_sec": start.seconds,
            "end_sec": end.seconds,
            "duration_sec": end.seconds - start.seconds
        })

    return pd.DataFrame(rows)

#### **Track Detection**

In [ ]:
def bbox_detection(input_video, shots_data) -> Dict:
    if Path(f"./working/{input_video['filename']}/video_only.mp4").exists():
        video_path = f"./working/{input_video['filename']}/video_only.mp4"
    else:
        video_path = input_video["path"]

    YOLO_MODEL = YOLO("./models/yolo26s.pt")

    FACE_DETECTOR = vision.FaceDetector.create_from_options(
        vision.FaceDetectorOptions(
            base_options=BaseOptions(model_asset_path="./models/blaze_face_full_range.tflite"),
            running_mode=vision.RunningMode.IMAGE,
            min_detection_confidence=0.5
        )
    )

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")

    shot_frame_tracks = {}
    for _, shot in shots_data.iterrows():
        shot_id = int(shot["shot_id"])
        start_frame = int(shot["start_frame"])
        end_frame = int(shot["end_frame"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        if hasattr(YOLO_MODEL, "predictor") and YOLO_MODEL.predictor is not None:
            YOLO_MODEL.predictor.trackers = None

        frame_tracks = {}
        for frame_idx in range(start_frame, end_frame):
            current_frame_tracks = []

            ret, frame = cap.read()
            if not ret:
                break

            # Detect Object and Tracking
            yolo_results = YOLO_MODEL.track(
                source=frame,
                tracker="bytetrack.yaml",
                persist=(frame_idx != start_frame),
                conf=0.25,
                iou=0.5,
                verbose=False,
                device=ACTIVE_DEVICE
            )
            yolo_boxes = yolo_results[0].boxes
            yolo_names = yolo_results[0].names

            if yolo_boxes.id is not None:
                for tid, box, conf, cid in zip(
                    yolo_boxes.id.int().cpu().tolist(),
                    yolo_boxes.xyxy.int().cpu().tolist(),
                    yolo_boxes.conf.cpu().tolist(),
                    yolo_boxes.cls.int().cpu().tolist()
                ):
                    if yolo_names[cid] == "person":
                        px1, py1, px2, py2 = box

                        person_crop = frame[py1:py2, px1:px2]
                        h, w = person_crop.shape[:2]

                        mp_results = FACE_DETECTOR.detect(mp.Image(
                            image_format=mp.ImageFormat.SRGB,
                            data=np.ascontiguousarray(cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)),
                        ))

                        if mp_results.detections:
                            target_cx, target_cy = w * 0.5, h * 0.2

                            # Pick the detection whose center is closest to the target head position
                            mp_detection = min(
                                mp_results.detections,
                                key=lambda d: (
                                    (d.bounding_box.origin_x + d.bounding_box.width * 0.5 - target_cx) ** 2
                                    + (d.bounding_box.origin_y + d.bounding_box.height * 0.5 - target_cy) ** 2
                                ),
                            )

                            mp_bbox = mp_detection.bounding_box
                            fx1, fy1 = max(0, mp_bbox.origin_x), max(0, mp_bbox.origin_y)
                            fx2, fy2 = min(w, fx1 + mp_bbox.width), min(h, fy1 + mp_bbox.height)

                            current_frame_tracks.append({
                                "track_id": tid,
                                "bbox": [px1, py1, px2, py2],
                                "conf": conf,
                                "cname": yolo_names[cid],

                                "face_bbox": [px1+fx1, py1+fy1, px1+fx2, py1+fy2],
                                "face_conf": mp_detection.categories[0].score,
                            })
                    else:
                        current_frame_tracks.append({
                            "track_id": tid,
                            "bbox": box,
                            "conf": conf,
                            "cname": yolo_names[cid]
                        })

            frame_tracks[frame_idx] = current_frame_tracks
        shot_frame_tracks[shot_id] = frame_tracks

    # Clean up
    del YOLO_MODEL
    del FACE_DETECTOR

    cap.release()
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    return shot_frame_tracks

#### **Seperate: Vocals and No Vocals**

In [ ]:
def demucs(input_video: Dict):
    if Path(f"./working/{input_video['filename']}/audio_only.wav").exists():
        audio_path = f"./working/{input_video['filename']}/audio_only.wav"
    else:
        print("Audio only path not found!")
        return None, None

    temp_dir = Path(f"./working/{input_video['filename']}/stems_temp")
    temp_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "demucs",
            "--two-stems",
            "vocals",
            "-n",
            "htdemucs",
            "-d",
            ACTIVE_DEVICE,
            "-o",
            str(temp_dir),
            audio_path,
        ],
        check=True,
        capture_output=True,
        text=True
    )

    vocals = f"./working/{input_video['filename']}/vocals.wav"
    no_vocals = f"./working/{input_video['filename']}/no_vocals.wav"

    shutil.move(f"./working/{input_video['filename']}/stems_temp/htdemucs/audio_only/vocals.wav", vocals)
    shutil.move(f"./working/{input_video['filename']}/stems_temp/htdemucs/audio_only/no_vocals.wav", no_vocals)
    shutil.rmtree(temp_dir)

    return (vocals, no_vocals)

#### **OCR: Text Detection**

In [ ]:
def text_detection(input_video: Dict, shots_df: pd.DataFrame, skip: int=1, langs: List=["en"]) -> pd.DataFrame:
    video_path = Path(f"./working/{input_video.get('filename', '')}/video_only.mp4")
    if not video_path.exists():
        print("Could not find VIDEO_ONLY file, using original video!")
        video_path = Path(input_video.get("path", ""))

    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    gpu_available = False if ACTIVE_DEVICE == "cpu" else True
    reader = easyocr.Reader(
        lang_list=langs,
        gpu=True,
        model_storage_directory="./models",
        download_enabled=True,
        verbose=False
    )

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return pd.DataFrame(
            columns=[
                "text",
                "shot_id",
                "start_s",
                "end_s",
                "duration_s",
                "confidence",
                "text_box_vertices",
            ]
        )

    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1920.0
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1080.0

    records = []

    for _, shot in shots_df.iterrows():
        s_id = shot["shot_id"]
        s_start_f = int(shot["start_frame"])
        s_end_f = int(shot["end_frame"])
        s_start_s = float(shot["start_sec"])
        s_end_s = float(shot["end_sec"])
        duration_s = round(s_end_s - s_start_s, 3)

        if skip == 1:
            sample_frames = [(s_start_f + s_end_f) // 2]
        else:
            step = (s_end_f - s_start_f) / (skip + 1)
            sample_frames = [int(s_start_f + step * (i + 1)) for i in range(skip)]

        shot_texts: Dict[str, Dict[str, Any]] = {}
        for f_idx in sample_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
            ret, frame = cap.read()
            if not ret or frame is None:
                continue

            # EasyOCR expects RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = reader.readtext(frame, paragraph=True)

            for item in results:
                if len(item) == 3:
                    bbox, text, conf = item
                    conf = float(conf)
                else:
                    bbox, text = item
                    conf = 1.0

                clean = text.strip()
                if len(clean) < 2 or conf < 0.35:
                    continue

                vertices = [(
                    round(float(pt[0]) / width, 4),
                    round(float(pt[1]) / height, 4),
                ) for pt in bbox]

                if clean not in shot_texts:
                    shot_texts[clean] = {
                        "text": clean,
                        "shot_id": s_id,
                        "start_s": s_start_s,
                        "end_s": s_end_s,
                        "duration_s": duration_s,
                        "confidence": conf,
                        "text_box_vertices": vertices,
                    }
                else:
                    shot_texts[clean]["confidence"] = max(
                        shot_texts[clean]["confidence"], conf
                    )

        records.extend(shot_texts.values())

    cap.release()
    del reader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    df = pd.DataFrame(records)
    df = df.sort_values(by="start_s").reset_index(drop=True)
    if df.empty:
        return pd.DataFrame(
            columns=[
                "text",
                "shot_id",
                "start_s",
                "end_s",
                "duration_s",
                "confidence",
                "text_box_vertices",
            ]
        )

    return df

#### **Transcribe Vocals**

In [ ]:
def transcribe_vocals(input_video: Dict):
    if Path(f"./working/{input_video['filename']}/vocals.wav").exists():
        vocals_path = f"./working/{input_video['filename']}/vocals.wav"
    else:
        print("No vocals path found!")
        return None

    compute_type = "float16" if ACTIVE_DEVICE == "cuda" else "int8"
    WHISPER_MODEL = WhisperModel("./models/whisper-medium", device=ACTIVE_DEVICE, compute_type=compute_type)

    segments, info = WHISPER_MODEL.transcribe(
        vocals_path,
        vad_filter=True,
        word_timestamps=True,
        temperature=0.0,
        condition_on_previous_text=False
    )

    transcript_df = []
    for idx, segment in enumerate(segments):
        words_data = [
            {
                "start": w.start,
                "end": w.end,
                "word": w.word.strip(),
                "probability": round(w.probability, 5),
            }
            for w in segment.words
        ] if segment.words else []

        transcript_df.append({
            "audio_segment_id": idx,
            "start_s": float(segment.start),
            "end_s": float(segment.end),
            "duration_s": float(segment.end - segment.start),
            "segment_transcript": segment.text.strip(),
            "words_data": words_data
        })

    # Clean up
    del WHISPER_MODEL
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    return pd.DataFrame(transcript_df)

#### **TalkNet**

##### **Preprocess Inputs**

In [ ]:
def preprocess_talknet_inputs(
    input_video: Dict,
    shot_tracks: Dict,
    shots_df: pd.DataFrame,
    min_track_len: int = 10,
    crop_scale: float = 0.40,
) -> List[Dict[str, Any]]:
    video_path = f"./working/{input_video['filename']}/video_only.mp4"
    if not Path(video_path).exists():
        video_path = input_video["path"]

    audio_path = f"./working/{input_video['filename']}/audio_only.wav"

    # 1. Load and resample audio once (44.1 kHz -> 16 kHz)
    sr, audio = wavfile.read(audio_path)
    if audio.ndim > 1:
        audio = audio[:, 0]
    if audio.dtype not in (np.float32, np.float64):
        audio = audio.astype(np.float32)

    target_sr = 16000
    if sr != target_sr:
        gcd = np.gcd(sr, target_sr)
        audio_16k = signal.resample_poly(
            audio, target_sr // gcd, sr // gcd
        )
    else:
        audio_16k = audio

    # 2. Extract global MFCC once across entire audio stream (100 Hz = 4 MFCC per video frame)[cite: 1]
    global_mfcc = python_speech_features.mfcc(
        audio_16k,
        samplerate=target_sr,
        numcep=13,
        winlen=0.025,
        winstep=0.010,
    )

    fps = float(input_video.get("fps", 25.0))
    processed_tracks = []
    cap = cv2.VideoCapture(video_path)

    for _, shot in shots_df.iterrows():
        s_id = int(shot["shot_id"])
        if s_id not in shot_tracks:
            continue

        frame_tracks = shot_tracks[s_id]

        track_buckets: Dict[int, Dict[str, list]] = {}
        for frame_idx, detections in frame_tracks.items():
            for det in detections:
                if det.get("cname") != "person" or det.get("face_bbox") is None:
                    continue
                tid = det["track_id"]
                if tid not in track_buckets:
                    track_buckets[tid] = {"frames": [], "face_bboxes": []}
                track_buckets[tid]["frames"].append(frame_idx)
                track_buckets[tid]["face_bboxes"].append(det["face_bbox"])

        valid_tracks = {}
        needed_frames = set()

        for tid, data in track_buckets.items():
            if len(data["frames"]) < min_track_len:
                continue

            f_arr = np.array(data["frames"])
            b_arr = np.array(data["face_bboxes"])

            interp_frames = np.arange(f_arr[0], f_arr[-1] + 1)
            interp_bboxes = []
            for coord_idx in range(4):
                interp_fn = interp1d(
                    f_arr,
                    b_arr[:, coord_idx],
                    kind="linear",
                    fill_value="extrapolate",
                )
                interp_bboxes.append(interp_fn(interp_frames))
            interp_bboxes = np.stack(interp_bboxes, axis=1)

            valid_tracks[tid] = {
                "frames": interp_frames,
                "bboxes": interp_bboxes,
            }
            needed_frames.update(interp_frames.tolist())

        if not valid_tracks:
            continue

        s_start_f = int(shot["start_frame"])
        s_end_f = int(shot["end_frame"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, s_start_f)

        frame_cache = {}
        for curr_f in range(s_start_f, s_end_f):
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            if curr_f in needed_frames:
                frame_cache[curr_f] = frame

        for tid, data in valid_tracks.items():
            frames_seq = data["frames"]
            bboxes_seq = data["bboxes"]

            dets = {"x": [], "y": [], "s": []}
            for det in bboxes_seq:
                dets["s"].append(
                    max((det[3] - det[1]), (det[2] - det[0])) / 2.0
                )
                dets["y"].append((det[1] + det[3]) / 2.0)
                dets["x"].append((det[0] + det[2]) / 2.0)

            k_size = (
                13
                if len(bboxes_seq) >= 13
                else (len(bboxes_seq) // 2 * 2 + 1)
            )
            if k_size >= 3:
                dets["s"] = signal.medfilt(dets["s"], kernel_size=k_size)
                dets["x"] = signal.medfilt(dets["x"], kernel_size=k_size)
                dets["y"] = signal.medfilt(dets["y"], kernel_size=k_size)

            video_features = []
            for i, f_idx in enumerate(frames_seq):
                img = frame_cache.get(f_idx)
                if img is None:
                    video_features.append(np.zeros((112, 112), dtype=np.uint8))
                    continue

                bs = dets["s"][i]
                bsi = int(bs * (1 + 2 * crop_scale))

                padded = np.pad(
                    img,
                    ((bsi, bsi), (bsi, bsi), (0, 0)),
                    "constant",
                    constant_values=(110, 110),
                )
                my = dets["y"][i] + bsi
                mx = dets["x"][i] + bsi

                face = padded[
                    int(my - bs) : int(my + bs * (1 + 2 * crop_scale)),
                    int(mx - bs * (1 + crop_scale)) : int(
                        mx + bs * (1 + crop_scale)
                    ),
                ]
                if face.size == 0:
                    face_224 = np.zeros((224, 224, 3), dtype=np.uint8)
                else:
                    face_224 = cv2.resize(face, (224, 224))

                face_gray = cv2.cvtColor(face_224, cv2.COLOR_BGR2GRAY)
                video_features.append(face_gray[56:168, 56:168])  #[cite: 1]

            video_features = np.array(video_features)

            # 3. Direct frame-indexed slicing into global MFCC[cite: 1]
            f_start = int(frames_seq[0])
            f_end = int(frames_seq[-1]) + 1
            a_start = f_start * 4
            a_end = f_end * 4

            mfcc_features = global_mfcc[a_start:a_end, :]

            # Strict 4:1 alignment
            v_len = min(video_features.shape[0], mfcc_features.shape[0] // 4)
            if v_len < min_track_len:
                continue

            aligned_video = video_features[:v_len, :, :]
            aligned_audio = mfcc_features[: v_len * 4, :]

            processed_tracks.append(
                {
                    "shot_id": s_id,
                    "track_id": tid,
                    "frames": frames_seq[:v_len],
                    "video_features": aligned_video,
                    "audio_features": aligned_audio,
                }
            )

    cap.release()
    return processed_tracks

##### **VisualEncoder**

In [ ]:
class ResNetLayer(nn.Module):
    def __init__(self, inplanes, outplanes, stride):
        super(ResNetLayer, self).__init__()
        self.conv1a = nn.Conv2d(inplanes, outplanes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1a = nn.BatchNorm2d(outplanes, momentum=0.01, eps=0.001)
        self.conv2a = nn.Conv2d(outplanes, outplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.stride = stride
        self.downsample = nn.Conv2d(inplanes, outplanes, kernel_size=(1,1), stride=stride, bias=False)
        self.outbna = nn.BatchNorm2d(outplanes, momentum=0.01, eps=0.001)

        self.conv1b = nn.Conv2d(outplanes, outplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1b = nn.BatchNorm2d(outplanes, momentum=0.01, eps=0.001)
        self.conv2b = nn.Conv2d(outplanes, outplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.outbnb = nn.BatchNorm2d(outplanes, momentum=0.01, eps=0.001)
        return

    def forward(self, inputBatch):
        batch = F.relu(self.bn1a(self.conv1a(inputBatch)))
        batch = self.conv2a(batch)
        if self.stride == 1:
            residualBatch = inputBatch
        else:
            residualBatch = self.downsample(inputBatch)
        batch = batch + residualBatch
        intermediateBatch = batch
        batch = F.relu(self.outbna(batch))

        batch = F.relu(self.bn1b(self.conv1b(batch)))
        batch = self.conv2b(batch)
        residualBatch = intermediateBatch
        batch = batch + residualBatch
        outputBatch = F.relu(self.outbnb(batch))
        return outputBatch

class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()
        self.layer1 = ResNetLayer(64, 64, stride=1)
        self.layer2 = ResNetLayer(64, 128, stride=2)
        self.layer3 = ResNetLayer(128, 256, stride=2)
        self.layer4 = ResNetLayer(256, 512, stride=2)
        self.avgpool = nn.AvgPool2d(kernel_size=(4,4), stride=(1,1))

        return

    def forward(self, inputBatch):
        batch = self.layer1(inputBatch)
        batch = self.layer2(batch)
        batch = self.layer3(batch)
        batch = self.layer4(batch)
        outputBatch = self.avgpool(batch)
        return outputBatch

class GlobalLayerNorm(nn.Module):
    def __init__(self, channel_size):
        super(GlobalLayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.Tensor(1, channel_size, 1))  # [1, N, 1]
        self.beta = nn.Parameter(torch.Tensor(1, channel_size, 1))  # [1, N, 1]
        self.reset_parameters()

    def reset_parameters(self):
        self.gamma.data.fill_(1)
        self.beta.data.zero_()

    def forward(self, y):
        mean = y.mean(dim=1, keepdim=True).mean(dim=2, keepdim=True) #[M, 1, 1]
        var = (torch.pow(y-mean, 2)).mean(dim=1, keepdim=True).mean(dim=2, keepdim=True)
        gLN_y = self.gamma * (y - mean) / torch.pow(var + 1e-8, 0.5) + self.beta
        return gLN_y

class visualFrontend(nn.Module):
    def __init__(self):
        super(visualFrontend, self).__init__()
        self.frontend3D = nn.Sequential(
                            nn.Conv3d(1, 64, kernel_size=(5,7,7), stride=(1,2,2), padding=(2,3,3), bias=False),
                            nn.BatchNorm3d(64, momentum=0.01, eps=0.001),
                            nn.ReLU(),
                            nn.MaxPool3d(kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1))
                        )
        self.resnet = ResNet()
        return

    def forward(self, inputBatch):
        inputBatch = inputBatch.transpose(0, 1).transpose(1, 2)
        batchsize = inputBatch.shape[0]
        batch = self.frontend3D(inputBatch)

        batch = batch.transpose(1, 2)
        batch = batch.reshape(batch.shape[0]*batch.shape[1], batch.shape[2], batch.shape[3], batch.shape[4])
        outputBatch = self.resnet(batch)
        outputBatch = outputBatch.reshape(batchsize, -1, 512)
        outputBatch = outputBatch.transpose(1 ,2)
        outputBatch = outputBatch.transpose(1, 2).transpose(0, 1)
        return outputBatch

class DSConv1d(nn.Module):
    def __init__(self):
        super(DSConv1d, self).__init__()
        self.net = nn.Sequential(
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Conv1d(512, 512, 3, stride=1, padding=1,dilation=1, groups=512, bias=False),
            nn.PReLU(),
            GlobalLayerNorm(512),
            nn.Conv1d(512, 512, 1, bias=False),
            )

    def forward(self, x):
        out = self.net(x)
        return out + x

class visualTCN(nn.Module):
    def __init__(self):
        super(visualTCN, self).__init__()
        stacks = []
        for x in range(5):
            stacks += [DSConv1d()]
        self.net = nn.Sequential(*stacks) # Visual Temporal Network V-TCN

    def forward(self, x):
        out = self.net(x)
        return out

class visualConv1D(nn.Module):
    def __init__(self):
        super(visualConv1D, self).__init__()
        self.net = nn.Sequential(
            nn.Conv1d(512, 256, 5, stride=1, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 128, 1),
            )

    def forward(self, x):
        out = self.net(x)
        return out

##### **Audio Encoder**

In [ ]:
class SEBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, reduction=8):
        super(SEBasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.se = SELayer(planes, reduction)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.relu(out)
        out = self.bn1(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)
        return out

class SELayer(nn.Module):
    def __init__(self, channel, reduction=8):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
                nn.Linear(channel, channel // reduction),
                nn.ReLU(inplace=True),
                nn.Linear(channel // reduction, channel),
                nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class audioEncoder(nn.Module):
    def __init__(self, layers, num_filters, **kwargs):
        super(audioEncoder, self).__init__()
        block = SEBasicBlock
        self.inplanes   = num_filters[0]

        self.conv1 = nn.Conv2d(1, num_filters[0] , kernel_size=7, stride=(2, 1), padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters[0])
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(block, num_filters[0], layers[0])
        self.layer2 = self._make_layer(block, num_filters[1], layers[1], stride=(2, 2))
        self.layer3 = self._make_layer(block, num_filters[2], layers[2], stride=(2, 2))
        self.layer4 = self._make_layer(block, num_filters[3], layers[3], stride=(1, 1))
        out_dim = num_filters[3] * block.expansion

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = torch.mean(x, dim=2, keepdim=True)
        x = x.view((x.size()[0], x.size()[1], -1))
        x = x.transpose(1, 2)

        return x

##### **Attention Layer**

In [ ]:
class attentionLayer(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super(attentionLayer, self).__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout=dropout)

        self.linear1 = nn.Linear(d_model, d_model * 4)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_model * 4, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = F.relu

    def forward(self, src, tar):
        src = src.transpose(0, 1) # B, T, C -> T, B, C
        tar = tar.transpose(0, 1) # B, T, C -> T, B, C
        src2 = self.self_attn(tar, src, src, attn_mask=None,
                              key_padding_mask=None)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        src = src.transpose(0, 1) # T, B, C -> B, T, C
        return src

##### **Model Class**

In [ ]:
class talkNetModel(nn.Module):
    def __init__(self):
        super(talkNetModel, self).__init__()
        # Visual Temporal Encoder
        self.visualFrontend  = visualFrontend() # Visual Frontend
        # self.visualFrontend.load_state_dict(torch.load('visual_frontend.pt', map_location="cuda"))
        # for param in self.visualFrontend.parameters():
        #     param.requires_grad = False
        self.visualTCN       = visualTCN()      # Visual Temporal Network TCN
        self.visualConv1D    = visualConv1D()   # Visual Temporal Network Conv1d

        # Audio Temporal Encoder
        self.audioEncoder  = audioEncoder(layers = [3, 4, 6, 3],  num_filters = [16, 32, 64, 128])

        # Audio-visual Cross Attention
        self.crossA2V = attentionLayer(d_model = 128, nhead = 8)
        self.crossV2A = attentionLayer(d_model = 128, nhead = 8)

        # Audio-visual Self Attention
        self.selfAV = attentionLayer(d_model = 256, nhead = 8)

        self.criterion = nn.CrossEntropyLoss()
        self.FC = nn.Linear(256, 2)

    def lossAV(self, x, labels=None):
        x = x.squeeze(1)
        x = self.FC(x)
        if labels == None:
            predScore = x[:,1]
            predScore = predScore.t()
            predScore = predScore.view(-1).detach().cpu().numpy()
            return predScore
        else:
            nloss = self.criterion(x, labels)
            predScore = F.softmax(x, dim = -1)
            predLabel = torch.round(F.softmax(x, dim = -1))[:,1]
            correctNum = (predLabel == labels).sum().float()
            return nloss, predScore, predLabel, correctNum

    def forward_visual_frontend(self, x):
        B, T, W, H = x.shape
        x = x.view(B*T, 1, 1, W, H)
        x = (x / 255 - 0.4161) / 0.1688
        x = self.visualFrontend(x)
        x = x.view(B, T, 512)
        x = x.transpose(1,2)
        x = self.visualTCN(x)
        x = self.visualConv1D(x)
        x = x.transpose(1,2)
        return x

    def forward_audio_frontend(self, x):
        x = x.unsqueeze(1).transpose(2, 3)
        x = self.audioEncoder(x)
        return x

    def forward_cross_attention(self, x1, x2):
        x1_c = self.crossA2V(src = x1, tar = x2)
        x2_c = self.crossV2A(src = x2, tar = x1)
        return x1_c, x2_c

    def forward_audio_visual_backend(self, x1, x2):
        x = torch.cat((x1,x2), 2)
        x = self.selfAV(src = x, tar = x)
        x = torch.reshape(x, (-1, 256))
        return x

    def forward_audio_backend(self,x):
        x = torch.reshape(x, (-1, 128))
        return x

    def forward_visual_backend(self,x):
        x = torch.reshape(x, (-1, 128))
        return x

##### **TalkNet Inference**

In [ ]:
def load_talknet() -> talkNetModel:
    model = talkNetModel()

    ckpt = torch.load("./models/pretrain_TalkSet.model", map_location=ACTIVE_DEVICE)
    state_dict = ckpt.get("model", ckpt)

    clean_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "")

        # Skip training-only single-modality auxiliary heads
        if name.startswith("lossA.") or name.startswith("lossV."):
            continue

        # Map prefixed keys to top-level attributes in talkNetModel[cite: 2]
        if name.startswith("model."):
            name = name[len("model.") :]
        elif name.startswith("lossAV."):
            name = name[len("lossAV.") :]

        clean_dict[name] = v

    # strict=True ensures all weights match exactly without silent failures
    model.load_state_dict(clean_dict, strict=True)
    model.eval()
    return model

In [ ]:
def evaluate_all_tracks(model: torch.nn.Module, processed_tracks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    duration_set = [1, 1, 1, 2, 2, 2, 3, 3, 4, 5, 6]
    scored_tracks = []

    model.eval()
    with torch.no_grad():
        for trk in processed_tracks:
            video_feature = trk["video_features"]  # (T, 112, 112)
            audio_feature = trk["audio_features"]  # (4T, 13)
            length = video_feature.shape[0] / 25.0

            all_scores = []
            for duration in duration_set:
                batch_size = int(math.ceil(length / duration))
                scores = []
                for i in range(batch_size):
                    sub_a = audio_feature[
                        i * duration * 100 : (i + 1) * duration * 100, :
                    ]
                    sub_v = video_feature[
                        i * duration * 25 : (i + 1) * duration * 25, :, :
                    ]

                    if len(sub_a) == 0 or len(sub_v) == 0:
                        continue

                    input_a = (
                        torch.FloatTensor(sub_a).unsqueeze(0).to(ACTIVE_DEVICE)
                    )
                    input_v = (
                        torch.FloatTensor(sub_v).unsqueeze(0).to(ACTIVE_DEVICE)
                    )

                    embed_a = model.forward_audio_frontend(input_a)
                    embed_v = model.forward_visual_frontend(input_v)
                    embed_a, embed_v = model.forward_cross_attention(
                        embed_a, embed_v
                    )
                    out = model.forward_audio_visual_backend(embed_a, embed_v)

                    score = model.lossAV(out, labels=None)
                    scores.extend(score)

                all_scores.append(scores)

            # Average over all duration windows[cite: 1]
            min_len = min(len(s) for s in all_scores)
            trimmed = [s[:min_len] for s in all_scores]
            mean_scores = np.mean(np.array(trimmed), axis=0)

            # frame smoothing window[cite: 1]
            smoothed = []
            for idx in range(len(mean_scores)):
                window = mean_scores[
                    max(idx - 2, 0) : min(idx + 3, len(mean_scores))
                ]
                smoothed.append(float(np.mean(window)))

            scored_tracks.append(
                {
                    "shot_id": trk["shot_id"],
                    "track_id": trk["track_id"],
                    "frames": trk["frames"][: len(smoothed)],
                    "scores": np.array(smoothed),
                }
            )

    return scored_tracks

#### **Motion Detection**

In [ ]:
def compute_frame_motion_saliency(
    input_video: Dict, shots_df: pd.DataFrame,
    diff_thresh: int = 25, density_thresh: float = 0.12, decay_rate: float = 0.92,
) -> Dict[int, Dict[str, float]]:
    if Path(f"./working/{input_video['filename']}/video_only.mp4").exists():
        video_path = f"./working/{input_video['filename']}/video_only.mp4"
    else:
        video_path = input_video["path"]

    cap = cv2.VideoCapture(video_path)
    total_frames = input_video["total_frames"]
    src_w = int(input_video["width"])
    src_h = int(input_video["height"])
    frame_area = float(src_w * src_h)

    kernel_size = int(min(src_w, src_h) * 0.12)
    kernel = np.ones((kernel_size, kernel_size), dtype=np.float32) / (kernel_size * kernel_size)

    shot_starts = set(shots_df["start_frame"].astype(int).tolist())

    prev_gray = None
    motion_results = {}
    anchor_weight = 0.0
    anchor_cx = src_w / 2.0
    for frame_idx in range(total_frames):
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        # Reset state on hard cuts
        if frame_idx in shot_starts or prev_gray is None:
            prev_gray = gray_blurred
            anchor_cx = src_w / 2.0
            anchor_weight = 0.0
            motion_results[frame_idx] = {
                "motion_cx": round(anchor_cx, 2),
                "motion_weight": 0.0,
                "instant_density": 0.0,
            }
            continue

        # Grayscale difference + thresholding
        frame_diff = cv2.absdiff(gray_blurred, prev_gray)
        _, motion_mask = cv2.threshold(frame_diff, diff_thresh, 255, cv2.THRESH_BINARY)

        # 2D sliding window density
        density_map = cv2.filter2D(motion_mask.astype(np.float32), -1, kernel)
        _, max_val, _, max_loc = cv2.minMaxLoc(density_map)

        peak_cx, _ = max_loc
        instant_density = float(np.clip(max_val / 255.0, 0.0, 1.0))

        # Update or decay focal anchor
        if instant_density >= density_thresh:
            anchor_cx = float(peak_cx)
            anchor_weight = min(1.0, instant_density * 1.5)
        else:
            anchor_weight = max(0.0, anchor_weight * decay_rate)

        motion_results[frame_idx] = {
            "motion_cx": round(anchor_cx, 2),
            "motion_weight": round(anchor_weight, 3),
            "instant_density": round(instant_density, 3),
        }

        prev_gray = gray_blurred

    cap.release()
    return motion_results

### **Run**

In [ ]:
INPUT_VIDEO = init_input_video("./mmt.mp4")
INPUT_VIDEO

{'path': 'mmt.mp4',
 'parent_dir': '.',
 'filename': 'mmt',
 'extension': '.mp4',
 'fps': 25.0,
 'total_frames': 625,
 'width': 1920,
 'height': 1080,
 'duration': 25.0}

In [ ]:
AUDIO_ONLY = separate_audio(INPUT_VIDEO)
VIDEO_ONLY = separate_video(INPUT_VIDEO)

AUDIO_ONLY, VIDEO_ONLY

('./working/mmt/audio_only.wav', './working/mmt/video_only.mp4')

In [ ]:
SHOTS_DF = shot_detection(INPUT_VIDEO)
SHOTS_DF

INFO:pyscenedetect:Detecting scenes...


,shot_id,start_frame,end_frame,duration_frame,start_sec,end_sec,duration_sec
0,0,0,32,32,0.00,1.28,1.28
1,1,32,109,77,1.28,4.36,3.08
2,2,109,141,32,4.36,5.64,1.28
3,3,141,158,17,5.64,6.32,0.68
4,4,158,187,29,6.32,7.48,1.16
5,5,187,237,50,7.48,9.48,2.00
6,6,237,256,19,9.48,10.24,0.76
7,7,256,298,42,10.24,11.92,1.68
8,8,298,338,40,11.92,13.52,1.60
9,9,338,363,25,13.52,14.52,1.00


In [ ]:
SHOT_TRACKS = bbox_detection(INPUT_VIDEO, SHOTS_DF)

In [ ]:
MOTION_SAL = compute_frame_motion_saliency(INPUT_VIDEO, SHOTS_DF)

In [ ]:
VOCALS, NO_VOCALS = demucs(INPUT_VIDEO)
VOCALS, NO_VOCALS

('./working/mmt/vocals.wav', './working/mmt/no_vocals.wav')

In [ ]:
TRANSCRIPT_DF = transcribe_vocals(INPUT_VIDEO)
TRANSCRIPT_DF

,audio_segment_id,start_s,end_s,duration_s,segment_transcript,words_data
0,0,0.00,1.52,1.52,हमारी ले गौआ से क्या लाए।,"[{'start': 0.0, 'end': 0.54, 'word': 'हमारी', ..."
1,1,1.52,4.96,3.44,जिगी साथ तेरे लिए किन केर प्रोड़ार्ट।,"[{'start': 1.52, 'end': 2.26, 'word': 'जिगी', ..."
2,2,6.64,7.50,0.86,वोटिल कोईनूर।,"[{'start': 6.64, 'end': 6.98, 'word': 'वोटिल',..."
3,3,7.50,8.64,1.14,गौआ का टोब ड़ान चे.,"[{'start': 7.5, 'end': 7.76, 'word': 'गौआ', 'p..."
4,4,8.76,9.68,0.92,बहुत हाईफाई चे.,"[{'start': 8.76, 'end': 8.98, 'word': 'बहुत', ..."
5,5,10.12,13.30,3.18,"मुटा बाई, आपके लिए टोपी.","[{'start': 10.12, 'end': 10.58, 'word': 'मुटा'..."
6,6,14.20,14.92,0.72,लेटेश्ट।,"[{'start': 14.2, 'end': 14.92, 'word': 'लेटेश्..."
7,7,14.92,16.72,1.80,"और बकुर, तेरी ले ये.","[{'start': 14.92, 'end': 15.18, 'word': 'और', ..."
8,8,17.16,17.88,0.72,शूज।,"[{'start': 17.16, 'end': 17.88, 'word': 'शूज।'..."
9,9,17.88,20.96,3.08,सारे पैसे होटेल पे वडाओगे तो एसे ही गिफ्ट लाओगे,"[{'start': 17.88, 'end': 18.38, 'word': 'सारे'..."


In [ ]:
OCR_DF = text_detection(INPUT_VIDEO, SHOTS_DF)
OCR_DF

,text,shot_id,start_s,end_s,duration_s,confidence,text_box_vertices
0,HOTEL,3.0,5.64,6.32,0.68,1.0,"[(0.6245, 0.4343), (0.6516, 0.4343), (0.6516, ..."
1,KOHINOOI GOA,3.0,5.64,6.32,0.68,1.0,"[(0.6099, 0.4444), (0.6646, 0.4444), (0.6646, ..."
2,0) HOTEL KQHINOOR GOA,3.0,5.64,6.32,0.68,1.0,"[(0.2875, 0.5269), (0.338, 0.5269), (0.338, 0...."
3,HOTL,3.0,5.64,6.32,0.68,1.0,"[(0.5182, 0.6231), (0.538, 0.6231), (0.538, 0...."
4,KOHINOOR G04,3.0,5.64,6.32,0.68,1.0,"[(0.4016, 0.6324), (0.4401, 0.6324), (0.4401, ..."
5,KOHINOOI GOA SHOWER GEL,3.0,5.64,6.32,0.68,1.0,"[(0.4823, 0.6324), (0.5604, 0.6324), (0.5604, ..."
6,Motsturising JDY LOTION,3.0,5.64,6.32,0.68,1.0,"[(0.2792, 0.6398), (0.3635, 0.6398), (0.3635, ..."
7,SHAMPOO,3.0,5.64,6.32,0.68,1.0,"[(0.3922, 0.6657), (0.4521, 0.6657), (0.4521, ..."
8,Kotel KOHAdon Coa,3.0,5.64,6.32,0.68,1.0,"[(0.6005, 0.6074), (0.6568, 0.6074), (0.6568, ..."
9,Shower CAP Kuiia oa Mm,9.0,13.52,14.52,1.00,1.0,"[(0.1016, 0.6685), (0.2172, 0.6685), (0.2172, ..."


In [ ]:
TALKNET_INPUTS = preprocess_talknet_inputs(INPUT_VIDEO, SHOT_TRACKS, SHOTS_DF)

TALKNET_MODEL = load_talknet().to(ACTIVE_DEVICE)
SCORED_TRACKS = evaluate_all_tracks(TALKNET_MODEL, TALKNET_INPUTS)

del TALKNET_MODEL
clean_gpu_memory()

In [ ]:
# 1. Collect positive speech scores
speech_lookup = {}
for trk in SCORED_TRACKS:
    s_id = trk["shot_id"]
    t_id = trk["track_id"]
    for f_idx, score in zip(trk["frames"], trk["scores"]):
        val = float(score)
        if val > 0.0:
            speech_lookup[(s_id, int(f_idx), t_id)] = round(val, 3)

# 2. Update detection objects directly
for s_id, frames_data in SHOT_TRACKS.items():
    for f_idx, detections in frames_data.items():
        for det in detections:
            if det.get("cname") != "person":
                continue

            key = (s_id, int(f_idx), det["track_id"])
            if key in speech_lookup:
                det["speaking"] = True
                det["asd_score"] = speech_lookup[key]
            else:
                det["speaking"] = False
                det["asd_score"] = None

In [ ]:
def parse_ocr_entries(ocr_df: pd.DataFrame, fps: float, video_w: int) -> Dict[int, List[Dict[str, float]]]:
    def safe_parse_vertices(val):
        if isinstance(val, (list, tuple)):
            return val
        if isinstance(val, str):
            val_clean = val.strip()
            try:
                return ast.literal_eval(val_clean)
            except (ValueError, SyntaxError):
                # Extract float coordinate pairs: (x, y)
                matches = re.findall(r"\(\s*([0-9.]+)\s*,\s*([0-9.]+)\s*\)", val_clean)
                if matches:
                    return [(float(x), float(y)) for x, y in matches]
        return []

    ocr_by_frame = {}
    for _, row in ocr_df.iterrows():
        f_start = int(round(float(row["start_s"]) * fps))
        f_end = int(round(float(row["end_s"]) * fps))

        vertices = safe_parse_vertices(row["text_box_vertices"])
        if not vertices or len(vertices) < 2:
            continue

        xs = [v[0] * video_w for v in vertices]
        ys = [v[1] for v in vertices]

        min_x, max_x = min(xs), max(xs)
        box_w = max_x - min_x
        box_h = max(ys) - min(ys)
        cx = (min_x + max_x) / 2.0
        area = box_w * box_h

        entry = {
            "cx": cx,
            "box_w": box_w,
            "area": area,
            "confidence": float(row.get("confidence", 1.0)),
        }

        for f in range(f_start, f_end + 1):
            if f not in ocr_by_frame:
                ocr_by_frame[f] = []
            ocr_by_frame[f].append(entry)

    return ocr_by_frame

In [ ]:
def compute_crop_centers(
    shots_df: pd.DataFrame,
    shot_tracks: Dict[int, Dict[int, List[Dict[str, Any]]]],
    ocr_df: pd.DataFrame,
    motion_results: Dict[int, Dict[str, float]],
    video_width: int,

    fps: float = 25.0,
    hangover_frames: int = 10,
    motion_hold_frames: int = 8,
) -> pd.DataFrame:
    """
    Step 1: Discovers the true focal coordinate per frame without smoothing,

    handling speech priorities and action-over-text takeover.
    """
    records = []
    ocr_by_frame = parse_ocr_entries(ocr_df, fps, video_width)
    default_cx = video_width / 2.0

    for _, shot in shots_df.iterrows():
        s_id = int(shot["shot_id"])
        s_start = int(shot["start_frame"])
        s_end = int(shot["end_frame"])
        frames_data = shot_tracks.get(s_id, {})

        last_speaker_tid = None
        hangover_remaining = 0
        last_valid_cx = default_cx

        # Motion persistence state
        last_action_cx = None
        action_hold_remaining = 0

        for f_idx in range(s_start, s_end):
            persons = [
                d
                for d in frames_data.get(f_idx, [])
                if d.get("cname") == "person"
            ]
            active_speakers = [p for p in persons if p.get("speaking") is True]

            target_cx = None
            reason = ""
            speaker_tid = None

            # -------------------------------------------------------------
            # Hierarchy Level 1: Exactly 1 person on screen
            # -------------------------------------------------------------
            if len(persons) == 1:
                p = persons[0]
                box = p.get("face_bbox") or p["bbox"]
                target_cx = (box[0] + box[2]) / 2.0
                reason = "single_person"

                if p.get("speaking") is True:
                    last_speaker_tid = p["track_id"]
                    speaker_tid = p["track_id"]
                    hangover_remaining = hangover_frames
                elif hangover_remaining > 0:
                    hangover_remaining -= 1

            # -------------------------------------------------------------
            # Hierarchy Level 2: Multiple persons on screen
            # -------------------------------------------------------------
            elif len(persons) > 1:
                if active_speakers:
                    spk = max(
                        active_speakers, key=lambda x: x.get("asd_score", 0.0)
                    )
                    last_speaker_tid = spk["track_id"]
                    speaker_tid = spk["track_id"]
                    hangover_remaining = hangover_frames
                    box = spk.get("face_bbox") or spk["bbox"]
                    target_cx = (box[0] + box[2]) / 2.0
                    reason = f"active_speaker_{speaker_tid}"

                elif hangover_remaining > 0 and last_speaker_tid is not None:
                    held_spk = next(
                        (
                            p
                            for p in persons
                            if p["track_id"] == last_speaker_tid
                        ),
                        None,
                    )
                    if held_spk is not None:
                        hangover_remaining -= 1
                        box = held_spk.get("face_bbox") or held_spk["bbox"]
                        target_cx = (box[0] + box[2]) / 2.0
                        reason = f"speaker_hangover_{hangover_remaining}"
                        speaker_tid = last_speaker_tid
                    else:
                        hangover_remaining = 0

                # Multiple persons with no active speech -> Salient person fallback
                if target_cx is None:
                    largest_person = max(
                        persons,
                        key=lambda p: (p["bbox"][2] - p["bbox"][0])
                        * (p["bbox"][3] - p["bbox"][1]),
                    )
                    box = (
                        largest_person.get("face_bbox")
                        or largest_person["bbox"]
                    )
                    target_cx = (box[0] + box[2]) / 2.0
                    reason = "salient_person_fallback"

            # -------------------------------------------------------------
            # Hierarchy Level 3: Zero persons on screen (Objects / Action / OCR)
            # -------------------------------------------------------------
            else:
                hangover_remaining = 0
                last_speaker_tid = None

                motion = motion_results.get(f_idx, {})
                density = motion.get("instant_density", 0.0)
                m_weight = motion.get("motion_weight", 0.0)
                mcx = motion.get("motion_cx", default_cx)

                # 3A: Physical Action Interrupt (Hand reaching in, cap pulling out)
                # Motion strictly preempts static OCR
                if density >= 0.10 or m_weight >= 0.20:
                    target_cx = mcx
                    last_action_cx = mcx
                    action_hold_remaining = motion_hold_frames
                    reason = f"action_takeover_d{density:.2f}"

                # 3B: Motion Hold (Item was pulled out and is now being held)
                elif action_hold_remaining > 0 and last_action_cx is not None:
                    target_cx = last_action_cx
                    action_hold_remaining -= 1
                    reason = f"action_hold_{action_hold_remaining}"

                # 3C: Static Graphic / Title card (Only when there is no active motion)
                elif f_idx in ocr_by_frame:
                    boxes = ocr_by_frame[f_idx]
                    best_box = max(boxes, key=lambda b: b["area"])
                    target_cx = best_box["cx"]
                    reason = "zero_person_ocr"

                # 3D: Quiescent hold
                else:
                    target_cx = last_valid_cx
                    reason = "zero_person_inertia"

            last_valid_cx = target_cx
            records.append(
                {
                    "shot_id": s_id,
                    "frame_idx": f_idx,
                    "target_cx": round(target_cx, 2),
                    "speaker_tid": speaker_tid,
                    "reason": reason,
                }
            )

    return pd.DataFrame(records)

In [ ]:
CROP_TRAJECTORY_DF = compute_crop_centers(
    shots_df=SHOTS_DF,
    shot_tracks=SHOT_TRACKS,
    ocr_df=OCR_DF,
    motion_results=MOTION_SAL,
    video_width=INPUT_VIDEO["width"]
)

CROP_TRAJECTORY_DF

,shot_id,frame_idx,target_cx,speaker_tid,reason
0,0,0,1276.50,4.0,active_speaker_4
1,0,1,1275.50,4.0,active_speaker_4
2,0,2,1275.00,4.0,active_speaker_4
3,0,3,1272.00,4.0,active_speaker_4
4,0,4,1270.50,4.0,active_speaker_4
...,...,...,...,...,...
620,15,620,961.54,NaN,zero_person_ocr
621,15,621,961.54,NaN,zero_person_ocr
622,15,622,961.54,NaN,zero_person_ocr
623,15,623,961.54,NaN,zero_person_ocr


In [ ]:
CROP_TRAJECTORY_DF[CROP_TRAJECTORY_DF['shot_id'] == 9].to_csv(index=False)

'shot_id,frame_idx,target_cx,speaker_tid,reason\n9,338,306.05,,zero_person_ocr\n9,339,497.0,,action_takeover_d0.32\n9,340,491.0,,action_takeover_d0.38\n9,341,828.0,,action_takeover_d0.69\n9,342,367.0,,action_takeover_d0.46\n9,343,715.0,,action_takeover_d0.50\n9,344,464.0,,action_takeover_d0.39\n9,345,1915.0,,action_takeover_d0.23\n9,346,477.0,,action_takeover_d0.54\n9,347,317.0,,action_takeover_d0.61\n9,348,673.0,,action_takeover_d1.00\n9,349,842.0,,action_takeover_d1.00\n9,350,1432.0,,action_takeover_d0.88\n9,351,1455.0,,action_takeover_d0.68\n9,352,251.0,,action_takeover_d0.57\n9,353,1395.0,,action_takeover_d0.58\n9,354,1332.0,,action_takeover_d0.64\n9,355,1465.0,,action_takeover_d0.57\n9,356,1472.0,,action_takeover_d0.79\n9,357,1422.0,,action_takeover_d0.81\n9,358,1235.0,,action_takeover_d0.51\n9,359,727.0,,action_takeover_d0.24\n9,360,1326.0,,action_takeover_d0.18\n9,361,1323.0,,action_takeover_d0.34\n9,362,1404.0,,action_takeover_d0.43\n'

In [ ]:
from typing import Any, Dict, List
import numpy as np
import pandas as pd


def extract_frame_candidates_v2(
    f_idx: int,
    shot_rel_idx: int,
    frames_data: Dict[int, List[Dict[str, Any]]],
    ocr_by_frame: Dict[int, List[Dict[str, float]]],
    motion_results: Dict[int, Dict[str, float]],
    video_w: int,
    video_h: int,
) -> List[Dict[str, Any]]:
    candidates = []
    frame_area = float(video_w * video_h)
    frame_dets = frames_data.get(f_idx, [])

    # 1. Active Speakers & Visible Persons
    for det in frame_dets:
        if det.get("cname") != "person":
            continue
        tid = det["track_id"]
        box = det.get("face_bbox") or det["bbox"]
        cx = (box[0] + box[2]) / 2.0
        area = (det["bbox"][2] - det["bbox"][0]) * (
            det["bbox"][3] - det["bbox"][1]
        )
        area_ratio = area / frame_area

        if det.get("speaking") is True:
            score = 2.0 + max(0.0, det.get("asd_score", 0.0))
            tag = f"speaker_{tid}"
        else:
            score = 0.60 + 0.30 * min(1.0, area_ratio / 0.25)
            tag = f"person_{tid}"

        candidates.append(
            {"cx": cx, "score": score, "tag": tag, "type": "person"}
        )

    # 2. Prominent OCR (Decays over time once read)
    if f_idx in ocr_by_frame:
        # High score for the first 12 frames (~0.5s), then steadily decays
        read_decay = 1.0 if shot_rel_idx < 12 else max(0.15, 1.0 - (shot_rel_idx - 12) * 0.08)
        for box in ocr_by_frame[f_idx]:
            score = 0.85 * read_decay
            candidates.append(
                {"cx": box["cx"], "score": score, "tag": "ocr_text", "type": "ocr"}
            )

    # 3. Motion Saliency (Action Peak)
    motion = motion_results.get(f_idx, {})
    density = motion.get("instant_density", 0.0)
    weight = motion.get("motion_weight", 0.0)
    mcx = motion.get("motion_cx")

    if mcx is not None and (density >= 0.10 or weight >= 0.15):
        # Spatially filter out edge artifacts (keep inside 5% - 95% width)
        if 0.05 * video_w <= mcx <= 0.95 * video_w:
            has_speaker = any(
                c["type"] == "person" and "speaker" in c["tag"] for c in candidates
            )
            # Physical action in zero-person shots commands highest saliency
            motion_score = 0.50 if has_speaker else 1.50 * (density + 0.5 * weight)
            candidates.append(
                {
                    "cx": mcx,
                    "score": motion_score,
                    "tag": "action_motion",
                    "type": "motion",
                }
            )

    if not candidates:
        candidates.append(
            {
                "cx": video_w / 2.0,
                "score": 0.20,
                "tag": "frame_center",
                "type": "fallback",
            }
        )

    return candidates


def optimize_shot_focal_path_v2(
    shot_frames: List[int],
    frames_data: Dict[int, List[Dict[str, Any]]],
    ocr_by_frame: Dict[int, List[Dict[str, float]]],
    motion_results: Dict[int, Dict[str, float]],
    video_w: int,
    video_h: int,
    max_jump_cost: float = 4.0,
    switch_penalty: float = 1.2,
) -> List[Dict[str, Any]]:
    T = len(shot_frames)
    if T == 0:
        return []

    frame_candidates = [
        extract_frame_candidates_v2(
            f, t, frames_data, ocr_by_frame, motion_results, video_w, video_h
        )
        for t, f in enumerate(shot_frames)
    ]

    dp_cost = []
    backpointer = []

    init_costs = [-np.log(max(1e-4, c["score"])) for c in frame_candidates[0]]
    dp_cost.append(init_costs)
    backpointer.append([-1] * len(frame_candidates[0]))

    for t in range(1, T):
        prev_cands = frame_candidates[t - 1]
        curr_cands = frame_candidates[t]
        t_costs = []
        t_bp = []

        for curr_c in curr_cands:
            obs_cost = -np.log(max(1e-4, curr_c["score"]))
            best_prev_idx = -1
            best_total_cost = float("inf")

            for prev_idx, prev_c in enumerate(prev_cands):
                dx = abs(curr_c["cx"] - prev_c["cx"])

                # Huber/Linear transition cost capped at max_jump_cost
                norm_dx = dx / float(video_w)
                trans_cost = min(max_jump_cost, 6.0 * norm_dx)

                if curr_c["tag"] != prev_c["tag"]:
                    trans_cost += switch_penalty

                total_cost = dp_cost[t - 1][prev_idx] + obs_cost + trans_cost
                if total_cost < best_total_cost:
                    best_total_cost = total_cost
                    best_prev_idx = prev_idx

            t_costs.append(best_total_cost)
            t_bp.append(best_prev_idx)

        dp_cost.append(t_costs)
        backpointer.append(t_bp)

    best_last_idx = int(np.argmin(dp_cost[-1]))
    path_indices = [best_last_idx]

    for t in range(T - 1, 0, -1):
        prev_idx = backpointer[t][path_indices[-1]]
        path_indices.append(prev_idx)

    path_indices.reverse()

    chosen_path = []
    for t, cand_idx in enumerate(path_indices):
        c = frame_candidates[t][cand_idx]
        chosen_path.append(
            {
                "frame_idx": shot_frames[t],
                "target_cx": round(c["cx"], 2),
                "tag": c["tag"],
                "type": c["type"],
            }
        )

    return chosen_path

In [ ]:
def find_focal_points_viterbi(
    shots_df: pd.DataFrame,
    shot_tracks: Dict[int, Dict[int, List[Dict[str, Any]]]],
    ocr_df: pd.DataFrame,
    motion_results: Dict[int, Dict[str, float]],
    video_width: int,
    video_height: int,
    fps: float = 25.0,
) -> pd.DataFrame:
    ocr_by_frame = parse_ocr_entries(ocr_df, fps, video_width)
    all_records = []

    for _, shot in shots_df.iterrows():
        s_id = int(shot["shot_id"])
        s_start = int(shot["start_frame"])
        s_end = int(shot["end_frame"])
        shot_frames = list(range(s_start, s_end))

        frames_data = shot_tracks.get(s_id, {})

        shot_path = optimize_shot_focal_path_v2(
            shot_frames=shot_frames,
            frames_data=frames_data,
            ocr_by_frame=ocr_by_frame,
            motion_results=motion_results,
            video_w=video_width,
            video_h=video_height,
            max_jump_cost=4.0,
            switch_penalty=1.2,
        )

        for entry in shot_path:
            entry["shot_id"] = s_id
            all_records.append(entry)

    return pd.DataFrame(all_records)


# Run Pass 1 & 2
FOCAL_POINTS_DF = find_focal_points_viterbi(
    shots_df=SHOTS_DF,
    shot_tracks=SHOT_TRACKS,
    ocr_df=OCR_DF,
    motion_results=MOTION_SAL,
    video_width=INPUT_VIDEO["width"],
    video_height=INPUT_VIDEO["height"],
    fps=INPUT_VIDEO["fps"],
)

# Print Shot 9 transition
FOCAL_POINTS_DF[FOCAL_POINTS_DF["shot_id"] == 9].to_csv()

',frame_idx,target_cx,tag,type,shot_id\n338,338,306.05,ocr_text,ocr,9\n339,339,306.05,ocr_text,ocr,9\n340,340,306.05,ocr_text,ocr,9\n341,341,306.05,ocr_text,ocr,9\n342,342,306.05,ocr_text,ocr,9\n343,343,306.05,ocr_text,ocr,9\n344,344,306.05,ocr_text,ocr,9\n345,345,306.05,ocr_text,ocr,9\n346,346,306.05,ocr_text,ocr,9\n347,347,306.05,ocr_text,ocr,9\n348,348,306.05,ocr_text,ocr,9\n349,349,306.05,ocr_text,ocr,9\n350,350,306.05,ocr_text,ocr,9\n351,351,306.05,ocr_text,ocr,9\n352,352,251.0,action_motion,motion,9\n353,353,1395.0,action_motion,motion,9\n354,354,1332.0,action_motion,motion,9\n355,355,1465.0,action_motion,motion,9\n356,356,1472.0,action_motion,motion,9\n357,357,1422.0,action_motion,motion,9\n358,358,1235.0,action_motion,motion,9\n359,359,727.0,action_motion,motion,9\n360,360,1326.0,action_motion,motion,9\n361,361,1323.0,action_motion,motion,9\n362,362,1404.0,action_motion,motion,9\n'

In [ ]:
from typing import Dict
import numpy as np
import pandas as pd


def plan_camera_kinematics(
    focal_df: pd.DataFrame,
    video_dict: Dict,
    pan_window_frames: int = 14,
    cut_threshold_px: float = 350.0,
    still_deadband_px: float = 45.0,
) -> pd.DataFrame:
    """Pass 3: Kinematic Camera Director.

    Transforms raw focal points into stationary holds, smooth ease pans, or cuts.
    """
    video_w = int(video_dict["width"])
    video_h = int(video_dict["height"])
    crop_w = int(round(video_h * (9.0 / 16.0)))
    if crop_w % 2 != 0:
        crop_w += 1

    min_cx = crop_w / 2.0
    max_cx = video_w - (crop_w / 2.0)

    output_records = []

    for shot_id, group in focal_df.groupby("shot_id", sort=False):
        group = group.sort_values("frame_idx").copy()
        raw_targets = group["target_cx"].to_numpy(dtype=np.float64)
        tags = group["tag"].to_numpy()
        frames = group["frame_idx"].to_numpy()
        N = len(raw_targets)

        # -------------------------------------------------------------
        # Step 1: Remove Isolated 1-2 Frame Spikes (Temporal Median Filter)
        # -------------------------------------------------------------
        filtered_targets = raw_targets.copy()
        for i in range(N):
            # 5-frame local neighborhood
            w_start = max(0, i - 2)
            w_end = min(N, i + 3)
            window = raw_targets[w_start:w_end]
            # Discard single-frame outliers like 251.0 surrounded by 1300+
            med_val = np.median(window)
            if abs(raw_targets[i] - med_val) > 200.0:
                filtered_targets[i] = med_val

        # -------------------------------------------------------------
        # Step 2: Determine Camera Action (Hold, Ease Pan, or Cut)
        # -------------------------------------------------------------
        kinematic_cx = np.zeros(N, dtype=np.float64)
        camera_modes = [""] * N

        # Hard cut onto frame 0 of the shot
        kinematic_cx[0] = float(np.clip(filtered_targets[0], min_cx, max_cx))
        camera_modes[0] = "shot_start_cut"

        i = 1
        while i < N:
            prev_cx = kinematic_cx[i - 1]
            target_cx = float(np.clip(filtered_targets[i], min_cx, max_cx))
            spatial_delta = target_cx - prev_cx
            abs_delta = abs(spatial_delta)

            # Mode A: Stillness / Deadband Hold (Zero Wobble)
            if abs_delta <= still_deadband_px:
                kinematic_cx[i] = prev_cx
                camera_modes[i] = "camera_hold"
                i += 1

            # Mode B: Instant Whip Cut (Extreme Jump or Dialogue Swap)
            elif (
                abs_delta > cut_threshold_px
                and tags[i] != tags[i - 1]
                and "speaker" in tags[i]
            ):
                kinematic_cx[i] = target_cx
                camera_modes[i] = "instant_cut"
                i += 1

            # Mode C: Smooth Cinematic Pan (Cosine Ease-in / Ease-out)
            else:
                # Calculate pan length bounded by remaining shot frames
                t_window = min(pan_window_frames, N - i)
                if t_window <= 1:
                    kinematic_cx[i] = target_cx
                    camera_modes[i] = "pan_step"
                    i += 1
                else:
                    steps = np.arange(1, t_window + 1)
                    # S-curve interpolation factors from 0 to 1
                    ease_factors = 0.5 * (1.0 - np.cos(np.pi * steps / t_window))

                    kinematic_cx[i : i + t_window] = prev_cx + spatial_delta * ease_factors
                    for step_idx in range(i, i + t_window):
                        camera_modes[step_idx] = "smooth_pan"
                    i += t_window

        # Clamp all values to legal boundaries
        kinematic_cx = np.clip(kinematic_cx, min_cx, max_cx)

        for idx in range(N):
            output_records.append(
                {
                    "shot_id": shot_id,
                    "frame_idx": int(frames[idx]),
                    "target_cx": round(float(raw_targets[idx]), 2),
                    "smoothed_cx": round(float(kinematic_cx[idx]), 2),
                    "camera_mode": camera_modes[idx],
                    "tag": tags[idx],
                }
            )

    return pd.DataFrame(output_records)

In [ ]:
# 1. Execute the camera kinematic director
CROP_TRAJECTORY_DF = plan_camera_kinematics(
    focal_df=FOCAL_POINTS_DF,
    video_dict=INPUT_VIDEO,
    pan_window_frames=12,      # 0.48s smooth cosine ease pan
    cut_threshold_px=450.0,    # Threshold for instant snap
    still_deadband_px=45.0,    # Eliminates micro-wobble
)

In [ ]:
from pathlib import Path
import subprocess
from typing import Dict
import cv2
import pandas as pd


def render_vertical_video(
    input_video: Dict,
    trajectory_df: pd.DataFrame,
    output_w: int = 1080,
    output_h: int = 1920,
) -> str:
  source_path = input_video["path"]
  filename = input_video["filename"]

  work_dir = Path(f"./working/{filename}")
  work_dir.mkdir(parents=True, exist_ok=True)

  temp_video = str(work_dir / "temp_vertical_muted.mp4")
  final_output = str(work_dir / "master_vertical_25s.mp4")

  cap = cv2.VideoCapture(source_path)
  fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
  src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

  # Fix: Ensure crop_w is always an even number
  crop_w = int(round(src_h * (9.0 / 16.0)))
  if crop_w % 2 != 0:
    crop_w += 1
  half_crop_w = crop_w // 2

  # Map frame -> smoothed center coordinate
  cx_lookup = dict(
      zip(trajectory_df["frame_idx"].astype(int), trajectory_df["smoothed_cx"])
  )

  fourcc = cv2.VideoWriter_fourcc(*"mp4v")
  out = cv2.VideoWriter(temp_video, fourcc, fps, (output_w, output_h))

  frame_idx = 0
  while True:
    ret, frame = cap.read()
    if not ret or frame is None:
      break

    cx = int(round(cx_lookup.get(frame_idx, src_w / 2.0)))
    x1 = max(0, cx - half_crop_w)
    x2 = x1 + crop_w

    if x2 > src_w:
      x2 = src_w
      x1 = src_w - crop_w

    cropped = frame[0:src_h, x1:x2]
    resized = cv2.resize(
        cropped, (output_w, output_h), interpolation=cv2.INTER_LANCZOS4
    )
    out.write(resized)
    frame_idx += 1

  cap.release()
  out.release()

  # Fix: Added -movflags +faststart for Mac playback, -c:a aac for audio compatibility, and captured error output
  cmd = [
      "ffmpeg",
      "-y",
      "-i",
      temp_video,
      "-i",
      source_path,
      "-map",
      "0:v:0",
      "-map",
      "1:a:0?",
      "-c:v",
      "libx264",
      "-pix_fmt",
      "yuv420p",
      "-movflags",
      "+faststart",
      "-c:a",
      "aac",
      "-b:a",
      "192k",
      "-shortest",
      final_output,
  ]

  res = subprocess.run(cmd, capture_output=True, text=True)
  Path(temp_video).unlink(missing_ok=True)

  if res.returncode != 0:
    raise RuntimeError(f"FFmpeg failed:\n{res.stderr}")

  return final_output

In [ ]:
render_vertical_video(INPUT_VIDEO, CROP_TRAJECTORY_DF)

'working/mmt/master_vertical_25s.mp4'

In [ ]:
CROP_TRAJECTORY_DF[CROP_TRAJECTORY_DF["shot_id"] == 9]

### **Render Videos**

In [ ]:
def _render_bbox(
    input_video: Dict, shot_frame_tracks: Dict, shots_data: pd.DataFrame
):
    if Path(f"./working/{input_video['filename']}/video_only.mp4").exists():
        video_path = f"./working/{input_video['filename']}/video_only.mp4"
    else:
        video_path = input_video["path"]

    output_path = f"./working/{input_video['filename']}/annotated.mp4"

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    for _, shot in shots_data.iterrows():
        shot_id = int(shot["shot_id"])
        start_frame = int(shot["start_frame"])
        end_frame = int(shot["end_frame"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frame_tracks = shot_frame_tracks[shot_id]
        for frame_idx in range(start_frame, end_frame):
            ret, frame = cap.read()
            if not ret:
                break

            cv2.putText(
                frame,
                f"Shot: #{shot_id} | Frame: {frame_idx}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 255),
                2,
            )

            for trk in frame_tracks.get(frame_idx, []):
                x1, y1, x2, y2 = trk["bbox"]
                cname = trk["cname"]
                tid = trk["track_id"]
                is_speaking = trk.get("speaking", False)
                asd_score = trk.get("asd_score")

                # Highlight speaking person in bright green, non-speaking in red/orange
                if cname == "person":
                    box_color = (0, 255, 0) if is_speaking else (0, 140, 255)
                    label = f"Speaker #{tid} ({asd_score:+.2f})" if is_speaking else f"Person #{tid}"
                else:
                    box_color = (255, 0, 255)
                    label = f"{cname} #{tid}" if tid is not None else cname

                # Draw Face Bounding Box
                if cname == "person" and trk.get("face_bbox") is not None:
                    fx1, fy1, fx2, fy2 = trk.get("face_bbox")
                    face_color = (0, 255, 0) if is_speaking else (0, 0, 255)
                    cv2.rectangle(frame, (fx1, fy1), (fx2, fy2), face_color, 2)

                # Draw Person Bounding Box
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(
                    frame,
                    label,
                    (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    box_color,
                    2,
                )

            out.write(frame)

    cap.release()
    out.release()

    return output_path

# _render_bbox(INPUT_VIDEO, SHOT_TRACKS, SHOTS_DF)

'./working/mmt/annotated.mp4'

In [ ]:
def _render_motion_saliency(
    input_video: Dict, motion_results: Dict, shots_data: pd.DataFrame,
):
    if Path(f"./working/{input_video['filename']}/video_only.mp4").exists():
        video_path = f"./working/{input_video['filename']}/video_only.mp4"
    else:
        video_path = input_video["path"]

    output_path = f"./working/{input_video['filename']}/annotated_motion.mp4"

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Adaptive box size matching the 12% kernel calculation
    box_dim = max(15, int(min(w, h) * 0.12))
    half_box = box_dim // 2

    out = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    for _, shot in shots_data.iterrows():
        shot_id = int(shot["shot_id"])
        start_frame = int(shot["start_frame"])
        end_frame = int(shot["end_frame"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        for frame_idx in range(start_frame, end_frame):
            ret, frame = cap.read()
            if not ret:
                break

            motion = motion_results.get(
                frame_idx,
                {"motion_cx": w / 2.0, "motion_weight": 0.0, "instant_density": 0.0},
            )

            cx = int(round(motion["motion_cx"]))
            weight = motion["motion_weight"]
            density = motion["instant_density"]

            # Highlight active movement in cyan; decaying hold in orange
            color = (0, 255, 255) if density > 0.12 else (0, 165, 255)

            # Draw focal line and anchor point
            cv2.line(frame, (cx, 0), (cx, h), color, 2)
            cv2.circle(frame, (cx, h // 2), 10, color, -1)

            # Draw focal box
            x1 = max(0, cx - half_box)
            x2 = min(w, cx + half_box)
            y1 = max(0, (h // 2) - half_box)
            y2 = min(h, (h // 2) + half_box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"Shot #{shot_id} | F: {frame_idx} | Cx: {cx} | Weight: {weight:.2f} | Density: {density:.2f}"
            cv2.putText(
                frame,
                label,
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2,
            )

            out.write(frame)

    cap.release()
    out.release()

    return output_path

_render_motion_saliency(INPUT_VIDEO, MOTION_SAL, SHOTS_DF)

'./working/mmt/annotated_motion.mp4'

### **Scrap**

In [ ]:
# def compute_raw_crop_centers(
#     shots_df: pd.DataFrame,
#     shot_tracks: Dict[int, Dict[int, List[Dict[str, Any]]]],
#     ocr_df: pd.DataFrame,
#     video_width: int,
#     crop_width: int,
#     fps: float = 25.0,
#     hangover_frames: int = 10,
# ) -> pd.DataFrame:
#     records = []
#     ocr_by_frame = parse_ocr_entries(ocr_df, fps, video_width)

#     min_cx = crop_width / 2.0
#     max_cx = video_width - (crop_width / 2.0)
#     default_cx = video_width / 2.0

#     for _, shot in shots_df.iterrows():
#         s_id = int(shot["shot_id"])
#         s_start = int(shot["start_frame"])
#         s_end = int(shot["end_frame"])

#         frames_data = shot_tracks.get(s_id, {})

#         # Intra-shot state
#         last_speaker_tid = None
#         hangover_remaining = 0
#         last_valid_cx = default_cx

#         for f_idx in range(s_start, s_end):
#             persons = [d for d in frames_data.get(f_idx, []) if d.get("cname") == "person"]
#             active_speakers = [p for p in persons if p.get("speaking") is True]

#             target_cx = None
#             selection_reason = ""

#             # -------------------------------------------------------------
#             # Hierarchy Level 1: Exactly 1 person on screen
#             # -------------------------------------------------------------
#             if len(persons) == 1:
#                 p = persons[0]
#                 if p.get("face_bbox") is not None:
#                     target_cx = (p["face_bbox"][0] + p["face_bbox"][2]) / 2.0
#                     selection_reason = "single_person_face"
#                 else:
#                     target_cx = (p["bbox"][0] + p["bbox"][2]) / 2.0
#                     selection_reason = "single_person_body"

#                 if p.get("speaking") is True:
#                     last_speaker_tid = p["track_id"]
#                     hangover_remaining = hangover_frames
#                 elif hangover_remaining > 0:
#                     hangover_remaining -= 1

#             # -------------------------------------------------------------
#             # Hierarchy Level 2: Multiple persons on screen
#             # -------------------------------------------------------------
#             elif len(persons) > 1:
#                 # 2A: Active speaker present
#                 if len(active_speakers) > 0:
#                     chosen_spk = max(active_speakers, key=lambda x: x.get("asd_score", 0.0))
#                     last_speaker_tid = chosen_spk["track_id"]
#                     hangover_remaining = hangover_frames

#                     if chosen_spk.get("face_bbox") is not None:
#                         target_cx = (chosen_spk["face_bbox"][0] + chosen_spk["face_bbox"][2]) / 2.0
#                     else:
#                         target_cx = (chosen_spk["bbox"][0] + chosen_spk["bbox"][2]) / 2.0
#                     selection_reason = "active_speaker"

#                 # 2B: Active speaker hangover hold
#                 elif hangover_remaining > 0 and last_speaker_tid is not None:
#                     held_spk = next((p for p in persons if p["track_id"] == last_speaker_tid), None)
#                     if held_spk is not None:
#                         hangover_remaining -= 1
#                         if held_spk.get("face_bbox") is not None:
#                             target_cx = (held_spk["face_bbox"][0] + held_spk["face_bbox"][2]) / 2.0
#                         else:
#                             target_cx = (held_spk["bbox"][0] + held_spk["bbox"][2]) / 2.0
#                         selection_reason = f"speaker_hangover_{hangover_remaining}"
#                     else:
#                         hangover_remaining = 0

#                 # 2C: Multiple persons, no speaker -> OCR fallback
#                 if target_cx is None and f_idx in ocr_by_frame:
#                     boxes = ocr_by_frame[f_idx]
#                     primary_box = max(boxes, key=lambda b: b["area"])
#                     target_cx = primary_box["cx"]
#                     selection_reason = "multi_person_ocr_fallback"

#                 # 2D: Multiple persons, no speaker, no OCR -> Salient person (largest body)
#                 if target_cx is None:
#                     largest_person = max(persons, key=lambda p: (p["bbox"][2] - p["bbox"][0]) * (p["bbox"][3] - p["bbox"][1]))
#                     if largest_person.get("face_bbox") is not None:
#                         target_cx = (largest_person["face_bbox"][0] + largest_person["face_bbox"][2]) / 2.0
#                     else:
#                         target_cx = (largest_person["bbox"][0] + largest_person["bbox"][2]) / 2.0
#                     selection_reason = "multi_person_salient_fallback"

#             # -------------------------------------------------------------
#             # Hierarchy Level 3: Zero persons on screen
#             # -------------------------------------------------------------
#             else:
#                 hangover_remaining = 0
#                 last_speaker_tid = None

#                 # 3A: OCR title card / graphics
#                 if f_idx in ocr_by_frame:
#                     boxes = ocr_by_frame[f_idx]
#                     primary_box = max(boxes, key=lambda b: b["area"])
#                     target_cx = primary_box["cx"]
#                     selection_reason = "zero_person_ocr"
#                 # 3B: Pure b-roll -> Hold prior position (or frame center)
#                 else:
#                     target_cx = last_valid_cx
#                     selection_reason = "zero_person_inertia_center"

#             # Clamp crop window to keep 9:16 frame inside original bounds [min_cx, max_cx]
#             clamped_cx = float(np.clip(target_cx, min_cx, max_cx))
#             last_valid_cx = clamped_cx

#             records.append({
#                 "shot_id": s_id,
#                 "frame_idx": f_idx,
#                 "raw_cx": round(target_cx, 2),
#                 "clamped_cx": round(clamped_cx, 2),
#                 "reason": selection_reason,
#             })

#     return pd.DataFrame(records)

In [ ]:
# def smooth_crop_trajectories(
#     crop_df: pd.DataFrame,
#     alpha: float = 0.15,
# ) -> pd.DataFrame:
#     smoothed_records = []

#     for shot_id, group in crop_df.groupby("shot_id", sort=False):
#         group = group.sort_values("frame_idx").copy()
#         raw_vals = group["smoothed_cx"].to_numpy()

#         smoothed = np.empty_like(raw_vals)
#         # Instant snap at shot start boundary
#         curr = raw_vals[0]
#         smoothed[0] = curr

#         for i in range(1, len(raw_vals)):
#             curr = alpha * raw_vals[i] + (1.0 - alpha) * curr
#             smoothed[i] = curr

#         group["smoothed_cx"] = np.round(smoothed, 2)
#         smoothed_records.append(group)

#     return pd.concat(smoothed_records, ignore_index=True)


# SMOOTHED_TRAJECTORY_DF = smooth_crop_trajectories(
#     CROP_TRAJECTORY_DF, alpha=0.15
# )

In [ ]:
from base64 import b64encode
import matplotlib.pyplot as plt
from IPython.display import HTML

def _show(video_path: str, width: int = 640):
    video_path = Path(video_path)
    h264_path = video_path.parent / f"h264_{video_path.name}"

    subprocess.run(
        f"ffmpeg -y -i {video_path} -vcodec libx264 -pix_fmt yuv420p -f mp4 {h264_path}",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    mp4_bytes = open(h264_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

    return HTML(f"""
    <video width="{width}" controls autoplay muted>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

_show("/content/working/mmt/master_vertical_25s.mp4")

CalledProcessError: Command 'ffmpeg -y -i /content/working/mmt/master_vertical_25s.mp4 -vcodec libx264 -pix_fmt yuv420p -f mp4 /content/working/mmt/h264_master_vertical_25s.mp4' returned non-zero exit status 1.